# 01. DART 데이터 수집

이 노트북은 `company_master.csv`의 381개 기업-연도(127개 기업 × 2022, 2023, 2024)에 대해 OpenDART 사업보고서 원문을 수집합니다.

분석 단위는 `stock_code × fiscal_year`이며, `esg_year = fiscal_year + 1` 규칙으로 KCGS 평가연도와 연결합니다.

**이번 수집 섹션** (기존 II, IV, VI 외 VIII, XI 추가)

| 라벨 | 한국어 제목 | ESG 연결 |
|---|---|---|
| II   | 사업의 내용 | E, S |
| IV   | 이사의 경영진단 및 분석의견 | E, S, G |
| VI   | 이사회 등 회사의 기관에 관한 사항 | G |
| VIII | 임원 및 직원 등에 관한 사항 | S, G |
| XI   | 그 밖에 투자자 보호를 위하여 필요한 사항 | E, S, G (보충) |

**산출물**

- `data/processed/dart_reports/{stock_code}_{fiscal_year}.json` — 기업-연도별 5개 섹션 원문과 메타데이터 (총 381개 시도)
- `data/processed/dart_collection_status.csv` — `회사이름`, `esg_year`, `수집여부` 3열 (요구사양). 디버깅용 `실패사유`를 함께 저장합니다.


## 1. 환경 설정과 API 키 로딩

`.env`에서 `DART_API_KEY` 또는 `OPENDART_API_KEY`를 읽고, 프로젝트 루트를 자동 탐지합니다.


In [1]:
import io
import json
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from html import unescape
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').exists() or (candidate / 'data').exists():
            return candidate
    return Path.cwd()


def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


PROJECT_ROOT = find_project_root()
load_env_file(PROJECT_ROOT / '.env')

API_KEY = os.getenv('DART_API_KEY') or os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('.env에 DART_API_KEY 또는 OPENDART_API_KEY를 설정하세요.')

OPENDART_BASE = 'https://opendart.fss.or.kr/api'
REQUEST_TIMEOUT = 60
SLEEP_BETWEEN_CALLS = 0.30

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROCESSED_DIR / 'dart_reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

COMPANY_MASTER_PATH = RAW_DIR / 'company_master.csv'
CORP_CODE_PATH = RAW_DIR / 'CORPCODE.xml'
STATUS_CSV_PATH = PROCESSED_DIR / 'dart_collection_status.csv'

print('프로젝트 루트:', PROJECT_ROOT)
print('JSON 출력 폴더:', REPORT_DIR)
print('상태 CSV 경로:', STATUS_CSV_PATH)
print('API key 앞 8자:', API_KEY[:8] + '...')


프로젝트 루트: /Users/home1/esg-dart-project
JSON 출력 폴더: /Users/home1/esg-dart-project/data/processed/dart_reports
상태 CSV 경로: /Users/home1/esg-dart-project/data/processed/dart_collection_status.csv
API key 앞 8자: 42a1b845...


## 2. company_master 로딩

`stock_code`는 6자리 문자열로 보정합니다(앞자리 0 보존). 381행 / 127개 stock_code / `esg_year = fiscal_year + 1` 무결성 확인.


In [2]:
def load_company_master(path: Path = COMPANY_MASTER_PATH) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'})
    df['stock_code'] = df['stock_code'].str.replace(r'\.0$', '', regex=True).str.zfill(6)
    df['fiscal_year'] = df['fiscal_year'].astype(int)
    df['esg_year'] = df['esg_year'].astype(int)
    return df


company_master = load_company_master()

print(f'행 수: {len(company_master)}')
print(f'고유 stock_code: {company_master["stock_code"].nunique()}')
print(f'esg_year == fiscal_year + 1: {(company_master["esg_year"] == company_master["fiscal_year"] + 1).all()}')
company_master.head()


행 수: 381
고유 stock_code: 127
esg_year == fiscal_year + 1: True


,company_name,corp_code,stock_code,industry,fiscal_year,report_code,rcept_no,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year
0,삼성전자,<NA>,005930,전기전자,2022,11011,<NA>,한국ESG기준원,A,A,A+,B+,2023
1,삼성전자,<NA>,005930,전기전자,2023,11011,<NA>,한국ESG기준원,B+,B+,A,B,2024
2,삼성전자,<NA>,005930,전기전자,2024,11011,<NA>,한국ESG기준원,A,B+,A+,B+,2025
3,BYC,<NA>,001460,섬유/의류,2022,11011,<NA>,한국ESG기준원,D,D,D,D,2023
4,BYC,<NA>,001460,섬유/의류,2023,11011,<NA>,한국ESG기준원,D,D,D,C,2024


## 3. corp_code 매핑

OpenDART `corpCode.xml`(ZIP)을 받아 `data/raw/CORPCODE.xml`로 캐시한 뒤, `stock_code → corp_code` 매핑을 만듭니다.


In [3]:
def download_corp_code_xml(force: bool = False) -> Path:
    if CORP_CODE_PATH.exists() and not force:
        print(f'캐시 사용: {CORP_CODE_PATH}')
        return CORP_CODE_PATH

    response = requests.get(f'{OPENDART_BASE}/corpCode.xml', params={'crtfc_key': API_KEY}, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    if response.content[:2] != b'PK':
        raise RuntimeError(f'corpCode.xml ZIP 아님: {response.content[:200]!r}')
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extract('CORPCODE.xml', path=CORP_CODE_PATH.parent)
    print(f'저장: {CORP_CODE_PATH}')
    return CORP_CODE_PATH


def build_stock_to_corp_map() -> dict[str, str]:
    if not CORP_CODE_PATH.exists():
        download_corp_code_xml()
    tree = ET.parse(CORP_CODE_PATH)
    candidates: dict[str, dict[str, str]] = {}
    for item in tree.getroot().findall('list'):
        stock_code = (item.findtext('stock_code') or '').strip().zfill(6)
        corp_code = (item.findtext('corp_code') or '').strip()
        modify_date = (item.findtext('modify_date') or '').strip()
        if not stock_code.strip('0') or not corp_code:
            continue
        previous = candidates.get(stock_code)
        if previous is None or modify_date > previous['modify_date']:
            candidates[stock_code] = {'corp_code': corp_code, 'modify_date': modify_date}
    return {sc: row['corp_code'] for sc, row in candidates.items()}


download_corp_code_xml()
stock_to_corp = build_stock_to_corp_map()
missing_corp = sorted(set(company_master['stock_code']) - set(stock_to_corp))
print(f'전체 매핑 수: {len(stock_to_corp):,}')
print(f'company_master 중 누락: {len(missing_corp)}')


캐시 사용: /Users/home1/esg-dart-project/data/raw/CORPCODE.xml
전체 매핑 수: 3,965
company_master 중 누락: 0


## 4. 사업보고서 rcept_no 선택

`list.json`에서 `fiscal_year + 1` 공시연도의 사업보고서를 찾고, 보고서명에 대상 회계연도가 포함된 것을 우선합니다. `[기재정정]`을 원본보다 위, `[첨부정정]`은 뒤로 둡니다.

비-12월 결산 기업도 같은 함수에서 처리할 수 있도록 `company.json`으로 `acc_mt`(결산월)을 조회해 보고서명 매칭 패턴을 보정합니다(예: 3월 결산 기업은 fiscal_year=2022 → `(2023.03)`).


In [4]:
@dataclass
class ReportSelection:
    report_nm: str
    rcept_no: str
    rcept_dt: str
    corp_code: str
    stock_code: str
    fiscal_year: int
    acc_mt: int

    @property
    def viewer_url(self) -> str:
        return f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={self.rcept_no}'


def opendart_get_json(path: str, params: dict[str, Any], retries: int = 2) -> dict[str, Any]:
    request_params = {'crtfc_key': API_KEY, **params}
    last_error: Exception | None = None
    for attempt in range(retries + 1):
        try:
            response = requests.get(f'{OPENDART_BASE}/{path}', params=request_params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'OpenDART {path} 호출 실패: {last_error}')


def list_annual_reports(corp_code: str, disclosure_year: int) -> list[dict[str, Any]]:
    payload = opendart_get_json('list.json', {
        'corp_code': corp_code,
        'bgn_de': f'{disclosure_year}0101',
        'end_de': f'{disclosure_year}1231',
        'pblntf_detail_ty': 'A001',
        'page_count': '100',
    })
    status = payload.get('status')
    if status in {'013', '014'}:
        return []
    if status != '000':
        raise RuntimeError(f'list.json 에러: {status} {payload.get("message")}')
    return payload.get('list', [])


def fetch_acc_mt(corp_code: str) -> int:
    payload = opendart_get_json('company.json', {'corp_code': corp_code})
    if payload.get('status') != '000':
        return 12
    raw = (payload.get('acc_mt') or '12').strip()
    return int(raw) if raw else 12


def report_label_for_fiscal_year(fiscal_year: int, acc_mt: int) -> tuple[int, int]:
    if acc_mt == 12:
        return fiscal_year, 12
    return fiscal_year + 1, acc_mt


def report_priority(row: dict[str, Any]) -> tuple[int, int]:
    name = row.get('report_nm', '')
    if '[첨부정정]' in name:
        correction = 8
    elif '[기재정정]' in name:
        correction = 0
    elif name.replace(' ', '').startswith('사업보고서'):
        correction = 1
    else:
        correction = 4
    return (correction, -int(row.get('rcept_no', '0') or '0'))


def select_annual_report(corp_code: str, stock_code: str, fiscal_year: int, acc_mt: int) -> ReportSelection | None:
    target_year, target_month = report_label_for_fiscal_year(fiscal_year, acc_mt)
    candidates: list[dict[str, Any]] = []
    for disclosure_year in (fiscal_year + 1, fiscal_year + 2):
        try:
            candidates.extend(list_annual_reports(corp_code, disclosure_year))
        except RuntimeError:
            continue
    if not candidates:
        return None

    label_pattern = re.compile(rf'\({target_year}\.0?{target_month}[.)]')
    matched = [
        row for row in candidates
        if label_pattern.search(re.sub(r'\s+', '', row.get('report_nm', '')))
    ]
    if not matched:
        return None

    chosen = sorted(matched, key=report_priority)[0]
    return ReportSelection(
        report_nm=chosen.get('report_nm', ''),
        rcept_no=chosen.get('rcept_no', ''),
        rcept_dt=chosen.get('rcept_dt', ''),
        corp_code=corp_code,
        stock_code=stock_code,
        fiscal_year=fiscal_year,
        acc_mt=acc_mt,
    )


## 5. 보고서 XML 다운로드

`document.xml`은 ZIP을 반환합니다. ZIP 내부 XML들을 디코딩해 문자열 리스트로 반환합니다.


In [5]:
def decode_xml_bytes(raw: bytes) -> str:
    for encoding in ('utf-8', 'euc-kr', 'cp949'):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')


def fetch_report_xml_files(rcept_no: str) -> list[tuple[str, str]]:
    last_error: Exception | None = None
    for attempt in range(3):
        try:
            response = requests.get(
                f'{OPENDART_BASE}/document.xml',
                params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            content = response.content
            if content[:2] != b'PK':
                raise RuntimeError(f'document.xml 응답이 ZIP이 아님: {content[:200]!r}')
            xml_files = []
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                for name in zf.namelist():
                    if not name.lower().endswith('.xml'):
                        continue
                    xml_files.append((name, decode_xml_bytes(zf.read(name))))
            return xml_files
        except (requests.RequestException, zipfile.BadZipFile, RuntimeError) as exc:
            last_error = exc
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'document.xml 다운로드 실패: {last_error}')


## 6. 섹션 추출 (II, IV, VI, VIII, XI)

DART XML의 `SECTION-1` 단위로 제목을 읽고 다섯 섹션을 추출합니다.

| 섹션 | 제목 키워드 |
|---|---|
| II   | 사업의 내용 |
| IV   | 이사의 경영진단 |
| VI   | 이사회 |
| VIII | 임원 및 직원 |
| XI   | 그 밖에 |

VIII가 II로, XI가 VI로 잘못 매칭되는 것을 막기 위해 긴 패턴부터 검사합니다.


In [6]:
TARGET_SECTION_LABELS = ('II', 'IV', 'VI', 'VIII', 'XI')

ROMAN_NORMALIZE = str.maketrans({
    'Ⅰ': 'I', 'Ⅱ': 'II', 'Ⅲ': 'III', 'Ⅳ': 'IV', 'Ⅴ': 'V',
    'Ⅵ': 'VI', 'Ⅶ': 'VII', 'Ⅷ': 'VIII', 'Ⅸ': 'IX', 'Ⅹ': 'X', 'Ⅺ': 'XI', 'Ⅻ': 'XII',
    'ⅰ': 'I', 'ⅱ': 'II', 'ⅲ': 'III', 'ⅳ': 'IV', 'ⅴ': 'V',
    'ⅵ': 'VI', 'ⅶ': 'VII', 'ⅷ': 'VIII', 'ⅸ': 'IX', 'ⅹ': 'X', 'ⅺ': 'XI', 'ⅻ': 'XII',
})

SECTION_TITLE_PATTERNS: dict[str, re.Pattern[str]] = {
    'II':   re.compile(r'^II\.?\s*사업의?\s*내용'),
    'IV':   re.compile(r'^IV\.?\s*이사의?\s*경영진단'),
    'VI':   re.compile(r'^VI\.?\s*이사회'),
    'VIII': re.compile(r'^VIII\.?\s*임원\s*및?\s*직원'),
    'XI':   re.compile(r'^XI\.?\s*그\s*밖에'),
}


def local_name(tag: str) -> str:
    return tag.rsplit('}', 1)[-1].upper()


def direct_title_text(element: ET.Element) -> str:
    for child in list(element):
        if local_name(child.tag) == 'TITLE':
            return ''.join(child.itertext()).strip()
    return ''


def classify_section(title: str) -> str | None:
    compact = (title or '').translate(ROMAN_NORMALIZE)
    compact = re.sub(r'\s+', ' ', compact).strip()
    # 길이가 긴 라벨부터 검사: VIII가 II로, XI가 VI로 오매칭되지 않도록.
    for label in ('VIII', 'XI', 'VI', 'IV', 'II'):
        if SECTION_TITLE_PATTERNS[label].match(compact):
            return label
    return None


def normalize_text(text: str) -> str:
    text = unescape(text)
    text = re.sub(r'[ \t\r\f]+', ' ', text)
    text = re.sub(r'\n[ \t]*', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def element_to_text(element: ET.Element) -> str:
    raw = ET.tostring(element, encoding='unicode', method='xml')
    raw = re.sub(r'</(P|TITLE|TR|LI|H[1-6]|TABLE)>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<BR\s*/?>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<[^>]+>', ' ', raw)
    return normalize_text(raw)


def extract_sections_with_parser(xml_text: str) -> dict[str, str | None]:
    result: dict[str, str | None] = {label: None for label in TARGET_SECTION_LABELS}
    try:
        root = ET.fromstring(xml_text.encode('utf-8'))
    except ET.ParseError:
        return result
    for element in root.iter():
        if local_name(element.tag) != 'SECTION-1':
            continue
        label = classify_section(direct_title_text(element))
        if label and result[label] is None:
            result[label] = element_to_text(element)
    return result


def extract_sections_with_regex(xml_text: str) -> dict[str, str | None]:
    result: dict[str, str | None] = {label: None for label in TARGET_SECTION_LABELS}
    starts = [m.start() for m in re.finditer(r'<SECTION-1\b[^>]*>', xml_text, flags=re.IGNORECASE)]
    starts.append(len(xml_text))
    for i in range(len(starts) - 1):
        chunk = xml_text[starts[i]:starts[i + 1]]
        title_match = re.search(r'<TITLE\b[^>]*>(.*?)</TITLE>', chunk, flags=re.DOTALL | re.IGNORECASE)
        if not title_match:
            continue
        title = normalize_text(re.sub(r'<[^>]+>', ' ', title_match.group(1)))
        label = classify_section(title)
        if label and result[label] is None:
            text = re.sub(r'</(P|TITLE|TR|LI|H[1-6]|TABLE)>', '\n', chunk, flags=re.IGNORECASE)
            text = re.sub(r'<BR\s*/?>', '\n', text, flags=re.IGNORECASE)
            text = re.sub(r'<[^>]+>', ' ', text)
            result[label] = normalize_text(text)
    return result


def extract_target_sections(xml_files: list[tuple[str, str]]) -> dict[str, str | None]:
    merged: dict[str, str | None] = {label: None for label in TARGET_SECTION_LABELS}
    for _, xml_text in xml_files:
        parsed = extract_sections_with_parser(xml_text)
        fallback = extract_sections_with_regex(xml_text) if any(v is None for v in parsed.values()) else {}
        for label in TARGET_SECTION_LABELS:
            if merged[label] is None:
                merged[label] = parsed.get(label) or fallback.get(label)
    return merged


## 7. 기업-연도 한 행 처리

각 행은 다음 순서로 처리됩니다.

1. `stock_code → corp_code` 매핑
2. `company.json`에서 결산월 조회
3. `list.json`에서 사업보고서 선택
4. `document.xml` 다운로드
5. 5개 섹션 추출
6. JSON 저장 (`{stock_code}_{fiscal_year}.json`)

이미 정상 JSON이 있으면 API 호출을 생략합니다 (`skip_existing=True`).


In [7]:
def report_json_path(stock_code: str, fiscal_year: int) -> Path:
    return REPORT_DIR / f'{stock_code}_{fiscal_year}.json'


def write_json_atomic(path: Path, payload: dict[str, Any]) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp.replace(path)


def load_existing_json(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        return None


def existing_has_target_sections(payload: dict[str, Any]) -> bool:
    sections = payload.get('sections') or {}
    return all(isinstance(sections.get(label), str) and sections.get(label).strip() for label in TARGET_SECTION_LABELS)


def make_status_row(row: pd.Series, success: bool, reason: str = '') -> dict[str, Any]:
    return {
        '회사이름': row.get('company_name', ''),
        'esg_year': int(row.get('esg_year')),
        '수집여부': bool(success),
        '실패사유': reason,
    }


def process_firm_year(row: pd.Series, skip_existing: bool = True) -> dict[str, Any]:
    stock_code = str(row['stock_code']).zfill(6)
    fiscal_year = int(row['fiscal_year'])
    esg_year = int(row['esg_year'])
    company_name = row.get('company_name', '')
    output_path = report_json_path(stock_code, fiscal_year)

    if skip_existing:
        existing = load_existing_json(output_path)
        if existing and existing.get('status') == 'OK' and existing_has_target_sections(existing):
            return make_status_row(row, True, '')

    corp_code = stock_to_corp.get(stock_code)
    if not corp_code:
        return make_status_row(row, False, 'corp_code 없음')

    try:
        acc_mt = fetch_acc_mt(corp_code)
        time.sleep(SLEEP_BETWEEN_CALLS)
        selection = select_annual_report(corp_code, stock_code, fiscal_year, acc_mt)
    except Exception as exc:
        return make_status_row(row, False, f'list.json: {exc}')

    if selection is None:
        return make_status_row(row, False, f'사업보고서 미발견 (acc_mt={acc_mt})')

    try:
        xml_files = fetch_report_xml_files(selection.rcept_no)
        sections = extract_target_sections(xml_files)
    except Exception as exc:
        return make_status_row(row, False, f'document.xml: {exc}')

    missing = [label for label in TARGET_SECTION_LABELS if not sections.get(label)]
    success = len(missing) == 0

    payload = {
        'status': 'OK' if success else 'PARTIAL',
        'company_name': company_name,
        'stock_code': stock_code,
        'corp_code': corp_code,
        'fiscal_year': fiscal_year,
        'esg_year': esg_year,
        'acc_mt': acc_mt,
        'report_nm': selection.report_nm,
        'rcept_no': selection.rcept_no,
        'rcept_dt': selection.rcept_dt,
        'viewer_url': selection.viewer_url,
        'sections': sections,
        'missing_sections': missing,
    }
    write_json_atomic(output_path, payload)

    return make_status_row(row, success, '' if success else f'섹션 누락: {missing}')


## 8. 단건 테스트 (삼성전자 2024)

전체 실행 전 한 행만 처리해 모든 단계가 작동하는지 확인합니다.


In [8]:
def run_sample(stock_code: str = '005930', fiscal_year: int = 2024) -> dict[str, Any]:
    target = company_master[
        (company_master['stock_code'] == stock_code.zfill(6))
        & (company_master['fiscal_year'] == fiscal_year)
    ]
    if target.empty:
        raise ValueError(f'company_master에 {stock_code} {fiscal_year} 없음')
    return process_firm_year(target.iloc[0], skip_existing=False)


# 네트워크 호출이 발생합니다.
sample_status = run_sample('005930', 2024)
print(sample_status)


{'회사이름': '삼성전자', 'esg_year': 2025, '수집여부': True, '실패사유': ''}


## 9. 전체 381행 수집

성공/실패와 상관없이 모든 행을 시도하고 `dart_collection_status.csv`에 결과를 누적합니다.

산출 CSV 열: `회사이름`, `esg_year`, `수집여부` (요구사양). `실패사유`는 디버깅용으로 함께 저장합니다.


In [9]:
def collect_all(df: pd.DataFrame = company_master, skip_existing: bool = True) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    total = len(df)
    started_at = time.time()

    for offset, (_, row) in enumerate(df.iterrows(), start=1):
        try:
            status = process_firm_year(row, skip_existing=skip_existing)
        except Exception as exc:
            status = make_status_row(row, False, f'unhandled: {exc}')
        rows.append(status)
        marker = 'OK' if status['수집여부'] else 'FAIL'
        print(f'{offset:03d}/{total} {row["stock_code"]} {int(row["fiscal_year"])} {row.get("company_name", "")}: {marker} {status["실패사유"]}')
        time.sleep(SLEEP_BETWEEN_CALLS)

    status_df = pd.DataFrame(rows, columns=['회사이름', 'esg_year', '수집여부', '실패사유'])
    status_df.to_csv(STATUS_CSV_PATH, index=False, encoding='utf-8-sig')

    elapsed = time.time() - started_at
    print(f'\n완료: {elapsed:.1f}초, 성공 {int(status_df["수집여부"].sum())}/{len(status_df)}')
    print(f'저장: {STATUS_CSV_PATH}')
    return status_df


# 전체 381행 수집입니다. OpenDART API 호출이 많이 발생합니다.
status_df = collect_all(skip_existing=True)
status_df['수집여부'].value_counts(dropna=False)


001/381 005930 2022 삼성전자: OK 
002/381 005930 2023 삼성전자: OK 
003/381 005930 2024 삼성전자: OK 
004/381 001460 2022 BYC: OK 
005/381 001460 2023 BYC: OK 
006/381 001460 2024 BYC: OK 
007/381 005380 2022 현대자동차: OK 
008/381 005380 2023 현대자동차: OK 
009/381 005380 2024 현대자동차: OK 
010/381 000120 2022 CJ대한통운: OK 
011/381 000120 2023 CJ대한통운: OK 
012/381 000120 2024 CJ대한통운: OK 
013/381 097950 2022 CJ제일제당: OK 
014/381 097950 2023 CJ제일제당: OK 
015/381 097950 2024 CJ제일제당: OK 
016/381 001040 2022 CJ: OK 
017/381 001040 2023 CJ: OK 
018/381 001040 2024 CJ: OK 
019/381 035760 2022 CJ ENM: OK 
020/381 035760 2023 CJ ENM: OK 
021/381 035760 2024 CJ ENM: OK 
022/381 282330 2022 BGF리테일: OK 
023/381 282330 2023 BGF리테일: OK 
024/381 282330 2024 BGF리테일: OK 
025/381 051500 2022 CJ프레시웨이: OK 
026/381 051500 2023 CJ프레시웨이: OK 
027/381 051500 2024 CJ프레시웨이: OK 
028/381 000660 2022 SK하이닉스: OK 
029/381 000660 2023 SK하이닉스: OK 
030/381 000660 2024 SK하이닉스: OK 
031/381 035420 2022 NAVER: OK 
032/381 035420 2023 NAVER: OK 
033/3

수집여부
True    381
Name: count, dtype: int64

## 10. 결과 검증

성공 행의 JSON에 5개 섹션이 모두 채워졌는지, 실패 사유 분포가 어떤지 확인합니다.


In [10]:
status_df = pd.read_csv(STATUS_CSV_PATH)

print('전체 행:', len(status_df))
print('수집 성공:', int(status_df['수집여부'].sum()))
print('수집 실패:', int((~status_df['수집여부']).sum()))

print('\n실패 사유 상위 10개:')
fail_reasons = (
    status_df.loc[~status_df['수집여부'], '실패사유']
    .value_counts()
    .head(10)
)
print(fail_reasons.to_string())

section_stats = []
for path in sorted(REPORT_DIR.glob('*.json')):
    payload = json.loads(path.read_text(encoding='utf-8'))
    sections = payload.get('sections') or {}
    section_stats.append({
        'file': path.name,
        **{f'len_{label}': len(sections.get(label) or '') for label in TARGET_SECTION_LABELS},
    })
section_df = pd.DataFrame(section_stats)
print(f'\n저장된 JSON 수: {len(section_df)}')
print('섹션별 평균 글자 수:')
print(section_df[[c for c in section_df.columns if c.startswith('len_')]].mean().round(0).to_string())


전체 행: 381
수집 성공: 381
수집 실패: 0

실패 사유 상위 10개:
Series([], )

저장된 JSON 수: 381
섹션별 평균 글자 수:
len_II      44283.0
len_IV      18602.0
len_VI      15531.0
len_VIII    11866.0
len_XI      12194.0
